## Data preprocessing
Our input image size of the model will be 512x512 (ade20k dataset)
So will convert our images and mask both in that size

In [10]:
from dotenv import load_dotenv
load_dotenv()
import os
DATA_DIR = os.getenv('DATA_DIR')
ROOT_DIR = os.getenv('ROOT_DIR')
DATA_DIR, ROOT_DIR

('/scratch/janakv', '/home2/pronoy.patra/Segmango_project/segmango_ssh')

In [2]:
import pandas as pd 
train_df = pd.read_csv(ROOT_DIR + '/data/train_test_splits/train_split_1.csv')
val_df = pd.read_csv(ROOT_DIR + '/data/train_test_splits/val_split_1.csv')
test_df = pd.read_csv(ROOT_DIR + '/data/train_test_splits/test_split.csv')
train_df['image_name'], val_df['image_name'], test_df['image_name']


(0        01_01_01
 1        01_01_02
 2        01_01_03
 3        01_01_04
 4        01_01_05
           ...    
 739    N_02_20_04
 740    N_02_20_05
 741    N_02_20_06
 742    N_02_20_07
 743    N_02_20_08
 Name: image_name, Length: 744, dtype: object,
 0       01_05_01
 1       01_05_02
 2       01_05_03
 3       01_05_04
 4       01_05_05
          ...    
 83    N_02_03_04
 84    N_02_03_05
 85    N_02_03_06
 86    N_02_03_07
 87    N_02_03_08
 Name: image_name, Length: 88, dtype: object,
 0        01_08_01
 1        01_08_02
 2        01_08_03
 3        01_08_04
 4        01_08_05
           ...    
 155    N_02_13_04
 156    N_02_13_05
 157    N_02_13_06
 158    N_02_13_07
 159    N_02_13_08
 Name: image_name, Length: 160, dtype: object)

In [3]:
import os
import shutil
from tqdm import tqdm

def merge_folders(src_dir, dst_dir):
    """
    Copies all files from src_dir to dst_dir.
    """
    # Ensure destination directory exists
    if not os.path.exists(dst_dir):
        os.makedirs(dst_dir)
        print(f"Created destination directory: {dst_dir}")

    # List all files in the source
    files = os.listdir(src_dir)
    
    print(f"Merging {len(files)} files from {src_dir} to {dst_dir}...")
    
    for file_name in tqdm(files):
        src_path = os.path.join(src_dir, file_name)
        dst_path = os.path.join(dst_dir, file_name)
        
        # Only copy if it's a file (skips subdirectories)
        if os.path.isfile(src_path):
            shutil.copy(src_path, dst_path)

# Example Usage:
folder_a = DATA_DIR + '/Dataset_images_2024'
folder_b = DATA_DIR + '/Dataset_images_2025'
target_folder = DATA_DIR + '/Dataset_images_2024_2025'

merge_folders(folder_a, target_folder)
merge_folders(folder_b, target_folder)

Created destination directory: /scratch/janakv/Dataset_images_2024_2025
Merging 768 files from /scratch/janakv/Dataset_images_2024 to /scratch/janakv/Dataset_images_2024_2025...


  1%|          | 6/768 [00:00<00:13, 56.90it/s]

100%|██████████| 768/768 [01:23<00:00,  9.24it/s]


Merging 480 files from /scratch/janakv/Dataset_images_2025 to /scratch/janakv/Dataset_images_2024_2025...


100%|██████████| 480/480 [00:35<00:00, 13.66it/s]


In [4]:

folder_a = DATA_DIR + '/Dataset_annotations_2024'
folder_b = DATA_DIR + '/Dataset_annotations_2025'
target_folder = DATA_DIR + '/Dataset_annotations_2024_2025'

merge_folders(folder_a, target_folder)
merge_folders(folder_b, target_folder)

Created destination directory: /scratch/janakv/Dataset_annotations_2024_2025
Merging 768 files from /scratch/janakv/Dataset_annotations_2024 to /scratch/janakv/Dataset_annotations_2024_2025...


100%|██████████| 768/768 [00:01<00:00, 428.66it/s]


Merging 480 files from /scratch/janakv/Dataset_annotations_2025 to /scratch/janakv/Dataset_annotations_2024_2025...


100%|██████████| 480/480 [00:00<00:00, 622.55it/s]


In [5]:
import os
import json
import shutil
import numpy as np
import cv2
from tqdm import tqdm

# Configuration
# DATA_DIR = 'path/to/your/original/data'  # Update this
OUTPUT_ROOT = DATA_DIR+'/segformer_my_dataset'
CLASSES = ('background', 'flower', 'fruitlet')

# Create folder structure
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(OUTPUT_ROOT, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_ROOT, 'annotations', split), exist_ok=True)

def process_split(df, split_name):
    print(f"Processing {split_name} split...")
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        img_name = row['image_name']
        
        # 1. Paths
        src_img = os.path.join(DATA_DIR+ '/Dataset_images_2024_2025', f"{img_name}.jpg")
        src_json = os.path.join(DATA_DIR+ '/Dataset_annotations_2024_2025', f"{img_name}.json")
        
        dst_img = os.path.join(OUTPUT_ROOT, 'images', split_name, f"{img_name}.jpg")
        dst_mask = os.path.join(OUTPUT_ROOT, 'annotations', split_name, f"{img_name}.png")
        # print(dst_img, src_img)
        # 2. Copy Image
        if os.path.exists(src_img):
            shutil.copy(src_img, dst_img)
            # print(f"Copied image: {src_img} to {dst_img}")
        # 3. Convert JSON to PNG Mask
        if os.path.exists(src_json):
            with open(src_json, 'r') as f:
                data = json.load(f)
            
            # Create blank mask (Height, Width) - Note: COCO has 8000h x 6000w
            # Your JSON says "width": 6000, "height": 8000
            mask = np.zeros((8000, 6000), dtype=np.uint8)
            
            for ann in data.get('annotations', []):
                cat_id = ann['category_id'] # flower=1, fruitlet=2
                segmentation = ann['segmentation']
                
                for poly in segmentation:
                    # Reshape flat list [x1, y1, x2, y2...] to [[x1, y1], [x2, y2]...]
                    poly_pts = np.array(poly).reshape((-1, 2)).astype(np.int32)
                    cv2.fillPoly(mask, [poly_pts], color=cat_id)
            
            cv2.imwrite(dst_mask, mask)
            # print(f"Created mask: {dst_mask} from {src_json}")
# Run for all splits
process_split(train_df, 'train')
process_split(val_df, 'val')
process_split(test_df, 'test')

print("Dataset preparation complete!")
# total images should be 992

Processing train split...


100%|██████████| 744/744 [03:29<00:00,  3.55it/s]


Processing val split...


100%|██████████| 88/88 [00:50<00:00,  1.74it/s]


Processing test split...


100%|██████████| 160/160 [00:48<00:00,  3.27it/s]

Dataset preparation complete!


In [1]:
import numpy as np
from PIL import Image
import os
import glob

mask_dir = '/scratch/janakv/segformer_my_dataset/annotations/train/*.png'
masks = glob.glob(mask_dir)

found_values = set()
print("Scanning dataset masks for unique pixel IDs...")

for m_path in masks[:50]: # Scan first 50 masks to spot the pattern
    mask = np.array(Image.open(m_path))
    found_values.update(np.unique(mask))

print(f"\nUnique pixel values found in your masks: {list(found_values)}")
print("Expected values for 3 classes: [0, 1, 2]")

Scanning dataset masks for unique pixel IDs...

Unique pixel values found in your masks: [0, 1, 2]
Expected values for 3 classes: [0, 1, 2]


## Model config and training

In [11]:
data_root = DATA_DIR+'/segformer_my_dataset'
img_dir = 'images'
ann_dir = 'annotations'
classes = ('background','flower','fruitlet')
palette = [[0,0,0],[255,0,0], [0,255,0]]

In [12]:
from mmengine import Config
cfg = Config.fromfile('/home2/pronoy.patra/Segmango_project/segmango_ssh/models/segformer_training/mmsegmentation/configs/segformer/segformer_mit-b1_8xb2-160k_ade20k-512x512.py')
print(f'Config:\n{cfg.pretty_text}')

Config:
checkpoint = 'https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b1_20220624-02e5a6a1.pth'
crop_size = (
    512,
    512,
)
data_preprocessor = dict(
    bgr_to_rgb=True,
    mean=[
        123.675,
        116.28,
        103.53,
    ],
    pad_val=0,
    seg_pad_val=255,
    size=(
        512,
        512,
    ),
    std=[
        58.395,
        57.12,
        57.375,
    ],
    type='SegDataPreProcessor')
data_root = 'data/ade/ADEChallengeData2016'
dataset_type = 'ADE20KDataset'
default_hooks = dict(
    checkpoint=dict(by_epoch=False, interval=16000, type='CheckpointHook'),
    logger=dict(interval=50, log_metric_by_epoch=False, type='LoggerHook'),
    param_scheduler=dict(type='ParamSchedulerHook'),
    sampler_seed=dict(type='DistSamplerSeedHook'),
    timer=dict(type='IterTimerHook'),
    visualization=dict(type='SegVisualizationHook'))
default_scope = 'mmseg'
env_cfg = dict(
    cudnn_benchmark=True,
    dist_cfg=dict(backend='nccl'),
    mp_cf

In [ ]:
# Since we use only one GPU, BN is used instead of SyncBN
cfg.norm_cfg = dict(type='BN', requires_grad=True)
cfg.crop_size = (512, 512)
cfg.model.data_preprocessor.size = cfg.crop_size
cfg.model.backbone.norm_cfg = cfg.norm_cfg
cfg.model.decode_head.norm_cfg = cfg.norm_cfg
# cfg.model.auxiliary_head.norm_cfg = cfg.norm_cfg
# modify num classes of the model in decode/auxiliary head
cfg.model.decode_head.num_classes = 3
# cfg.model.auxiliary_head.num_classes = 3

# Modify dataset type and path
cfg.dataset_type = 'MangoSenseDataset'
cfg.data_root = data_root

cfg.train_dataloader.batch_size = 12

cfg.train_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations'),
    dict(type='RandomResize', scale=(512, 512), ratio_range=(0.5, 2.0), keep_ratio=True),
    dict(type='RandomCrop', crop_size=cfg.crop_size, cat_max_ratio=0.75),
    dict(type='RandomFlip', prob=0.3),
    dict(type='PackSegInputs')
]

cfg.test_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='Resize', scale=(512, 512), keep_ratio=True),
    # add loading annotation after ``Resize`` because ground truth
    # does not need to do resize data transform
    dict(type='LoadAnnotations'),
    dict(type='PackSegInputs')
]
# --- Update Train Dataloader ---
cfg.train_dataloader.dataset.type = cfg.dataset_type
cfg.train_dataloader.dataset.data_root = cfg.data_root
cfg.train_dataloader.dataset.pipeline = cfg.train_pipeline
# cfg.train_dataloader.dataset.ann_file = 'splits/train.txt'
# cfg.train_dataloader.dataset.reduce_zero_label=True

cfg.val_dataloader.dataset.type = cfg.dataset_type
cfg.val_dataloader.dataset.data_root = cfg.data_root
cfg.val_dataloader.dataset.pipeline = cfg.test_pipeline
# cfg.val_dataloader.dataset.ann_file = 'splits/val.txt'
# cfg.val_dataloader.dataset.reduce_zero_label=True


cfg.test_dataloader.dataset.type = cfg.dataset_type
cfg.test_dataloader.dataset.data_root = cfg.data_root
cfg.test_dataloader.dataset.pipeline = cfg.test_pipeline
cfg.train_dataloader.dataset.data_prefix = dict(
    img_path='images/train',      # Point to the actual images
    seg_map_path='annotations/train' # Point to the actual masks
)

# --- Update Val Dataloader ---
cfg.val_dataloader.dataset.data_prefix = dict(
    img_path='images/val', 
    seg_map_path='annotations/val'
)

# --- Update Test Dataloader ---
cfg.test_dataloader.dataset.data_prefix = dict(
    img_path='images/test', 
    seg_map_path='annotations/test'
)

cfg.train_dataloader.dataset.reduce_zero_label = False
cfg.val_dataloader.dataset.reduce_zero_label = False
cfg.test_dataloader.dataset.reduce_zero_label = False

# Ensure LoadAnnotations also knows not to reduce
cfg.train_pipeline[1].reduce_zero_label = False
cfg.test_pipeline[2].reduce_zero_label = False
# cfg.train_dataloader.dataset.ignore_index = 0
# cfg.val_dataloader.dataset.ignore_index = 0
# cfg.test_dataloader.dataset.ignore_index = 0

# cfg.log_processor = dict(window_size = 200, by_epoch=True)

cfg.evaluation = dict(interval=cfg.train_cfg.val_interval, metric=['mIoU', 'IoU'], save_best='mIoU')

# Load the pretrained weights
# cfg.load_from = '/home2/janakv/mmsegmentation/pspnet_r50-d8_512x1024_40k_cityscapes_20200605_003338-2966598c.pth'

# Set up working dir to save files and logs.
cfg.work_dir = './work_dirs/segformer_512_sbatch'


cfg.train_cfg.max_iters = 1244*50    #1 epoch=25805
cfg.train_cfg.val_interval = 2488
cfg.default_hooks.logger.interval = 50
cfg.default_hooks.checkpoint.interval = 2488
cfg.default_hooks.checkpoint.by_epoch = False
cfg.default_hooks.checkpoint.max_keep_ckpts = 1
cfg.default_hooks.checkpoint.save_best='mIoU'
cfg.default_hooks.checkpoint.rule='greater'
cfg.default_hooks.checkpoint.save_last = True

cfg.model.decode_head.loss_decode = dict(
    type='CrossEntropyLoss', 
    use_sigmoid=False, 
    loss_weight=1.0,
    # Background=0.1, Flower=1.0, Fruitlet=1.0
    # This makes the model "feel" 10x more pain when it misses a flower
    class_weight=[0.1, 1.0, 1.0] 
)

cfg.model.decode_head.sampler = dict(type='OHEMPixelSampler', thresh=0.7, min_kept=100000)

cfg['randomness'] = dict(seed=0)

# Let's have a look at the final config used for training
# print(f'Config:\n{cfg.pretty_text}')

In [14]:
output_config_path = ROOT_DIR +'/models/segformer_training/mangosense_configs/mango_sense_segformer_512.py'

# Save the modified cfg to a file
cfg.dump(output_config_path)


In [ ]:
# bash mmsegmentation/tools/dist_train.sh mangosense_configs/mango_sense_segformer_512.py 4
# python mmsegmentation/tools/train.py mangosense_configs/mango_sense_segformer_512.py

In [1]:
# debug_dataset.py

from mmengine.config import Config
from mmseg.registry import DATASETS
from mmseg.registry import TRANSFORMS
from mmengine.dataset import Compose

cfg = Config.fromfile(
    'mangosense_configs/mango_sense_segformer_512.py'
)

dataset_cfg = cfg.train_dataloader.dataset

dataset = DATASETS.build(dataset_cfg)

print("Dataset length:", len(dataset))

sample = dataset[0]

print(sample.keys())

inputs = sample['inputs']
data_sample = sample['data_samples']

print("Input shape:", inputs.shape)
print("Seg shape:", data_sample.gt_sem_seg.data.shape)

print("Unique labels:",
      data_sample.gt_sem_seg.data.unique())

/home2/pronoy.patra/miniconda3/envs/segmango/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home2/pronoy.patra/miniconda3/envs/segmango/lib/python3.8/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


TypeError: __init__() got an unexpected keyword argument 'reduce_zero_label'